# 03 Run and Evaluate

This notebook crafts `ECOHOME_SYSTEM_PROMPT`, instantiates the LangGraph agent, runs 10 test cases, stores transparency data in `test_results`, and generates an evaluation report. The test cases cover EV charging, thermostat settings, appliances, solar maximization, battery use, historical usage, and cost savings.

## System Prompt

The prompt below defines the Energy Advisor's role, steps, capabilities, recommendation style, and example questions.

In [1]:
from agent import ECOHOME_SYSTEM_PROMPT
print(ECOHOME_SYSTEM_PROMPT)


You are EcoHome Energy Advisor, an AI agent for smart-home energy optimization.
Your role is to analyze weather forecasts, electricity pricing, solar generation,
historical device usage, and energy-saving best practices to provide personalized,
data-driven recommendations.

Steps:
1. Understand whether the user asks about EV charging, HVAC, appliances, solar,
   costs, savings, environmental impact, or usage history.
2. Retrieve the relevant data using weather, pricing, solar, usage, savings, and
   energy-tip tools.
3. Recommend a schedule or strategy that minimizes cost, maximizes solar usage,
   and preserves user comfort.
4. Cite relevant knowledge-base tips and include concrete savings or CO2 estimates
   when available.

Example questions include EV charging optimization, thermostat settings during
price spikes, appliance scheduling, pool-pump timing, and reducing usage based on
history.



## End-to-End Evaluation

This cell runs the agent over 10 scenarios. Each `test_results` item includes the question, final response, tools used, response metrics, and tool-usage metrics.

In [2]:
from agent import Agent, ECOHOME_SYSTEM_PROMPT
from evaluation import evaluate_response, evaluate_tool_usage, generate_evaluation_report, display_evaluation_report
from models.energy import populate_sample_data
from tools import search_energy_tips

populate_sample_data()
search_energy_tips.invoke('initialize vector database for solar energy tips')

ecohome_agent = Agent(
    instructions=ECOHOME_SYSTEM_PROMPT,
)

test_cases = [
    {
        'id': 'EV-001',
        'question': 'When should I charge my electric car tomorrow to minimize cost and maximize solar power?',
        'expected_response': 'Charge during midday solar hours and avoid evening peak prices.',
        'expected_tools': ['search_energy_tips', 'get_weather_forecast', 'get_solar_generation', 'get_electricity_prices', 'get_energy_usage', 'calculate_savings'],
    },
    {
        'id': 'HVAC-002',
        'question': 'What temperature should I set my thermostat on Wednesday afternoon if electricity prices spike?',
        'expected_response': 'Pre-cool before peak prices and raise the thermostat by 2 to 4 degrees during spikes.',
        'expected_tools': ['search_energy_tips', 'get_weather_forecast', 'get_solar_generation', 'get_electricity_prices', 'get_energy_usage'],
    },
    {
        'id': 'HISTORY-003',
        'question': 'Suggest three ways I can reduce energy use based on my usage history.',
        'expected_response': 'Use history to identify high-cost devices and shift flexible loads.',
        'expected_tools': ['search_energy_tips', 'get_energy_usage'],
    },
    {
        'id': 'APPLIANCE-004',
        'question': 'How much can I save by running my dishwasher during off-peak hours?',
        'expected_response': 'Move dishwasher from peak to off peak and estimate savings.',
        'expected_tools': ['search_energy_tips', 'get_electricity_prices', 'get_energy_usage', 'calculate_savings'],
    },
    {
        'id': 'POOL-005',
        'question': 'What is the best time to run my pool pump this week based on the weather forecast?',
        'expected_response': 'Run the pool pump during solar production and lower price windows.',
        'expected_tools': ['search_energy_tips', 'get_weather_forecast', 'get_solar_generation', 'get_electricity_prices', 'get_energy_usage'],
    },
    {
        'id': 'BATTERY-006',
        'question': 'Should I use my home battery during evening peak pricing?',
        'expected_response': 'Discharge battery during peak evening pricing while preserving backup capacity.',
        'expected_tools': ['search_energy_tips', 'get_electricity_prices'],
    },
    {
        'id': 'SOLAR-007',
        'question': 'How can I maximize self-consumption of solar power with laundry and appliances?',
        'expected_response': 'Shift appliances to midday solar windows and automate flexible loads.',
        'expected_tools': ['search_energy_tips', 'get_weather_forecast', 'get_solar_generation', 'get_energy_usage'],
    },
    {
        'id': 'EV-008',
        'question': 'Compare EV charging at 6 PM versus noon for a 12 kWh session.',
        'expected_response': 'Noon is cheaper and aligns with solar; estimate savings for 12 kWh.',
        'expected_tools': ['search_energy_tips', 'get_electricity_prices', 'get_solar_generation', 'get_energy_usage', 'calculate_savings'],
    },
    {
        'id': 'HVAC-009',
        'question': 'What seasonal strategy should I use for summer HVAC costs?',
        'expected_response': 'Pre-cool, use blinds, and avoid afternoon peak HVAC operation.',
        'expected_tools': ['search_energy_tips', 'get_weather_forecast', 'get_energy_usage'],
    },
    {
        'id': 'HISTORY-010',
        'question': 'Which devices have the highest historical energy cost?',
        'expected_response': 'Query usage history and identify the highest-cost devices.',
        'expected_tools': ['search_energy_tips', 'get_energy_usage'],
    },
]

test_results = []
for case in test_cases:
    result = ecohome_agent.invoke(case['question'])
    response_eval = evaluate_response(case['question'], result['final_response'], case['expected_response'])
    tool_eval = evaluate_tool_usage(result['messages'], case['expected_tools'])
    test_results.append({
        'id': case['id'],
        'question': case['question'],
        'final_response': result['final_response'],
        'tools_used': result['tools_used'],
        'messages_list': result['messages'],
        'response_evaluation': response_eval,
        'tool_evaluation': tool_eval,
    })

len(test_results), test_results[0]

(10,
 {'id': 'EV-001',
  'question': 'When should I charge my electric car tomorrow to minimize cost and maximize solar power?',
  'final_response': "EcoHome Energy Advisor recommendation\nQuestion: When should I charge my electric car tomorrow to minimize cost and maximize solar power?\n\nData retrieved:\n- search_energy_tips: used successfully\n- get_weather_forecast: used successfully\n- get_solar_generation: used successfully\n- get_electricity_prices: used successfully\n- get_energy_usage: used successfully\n- calculate_savings: used successfully\n\nRecommendation:\n- Prioritize solar-heavy hours around 12, 11, 13:00 for flexible loads.\n- Use hour 0:00 when possible at $0.13/kWh and avoid hour 16:00 at $0.39/kWh.\n- Weather outlook is rainy with best solar window 11:00-13:00.\n- Estimated savings for the schedule shift: $2.16, with about 4.54 kg CO2 avoided.\n\nRelevant knowledge-base citations:\n- tip_hvac_optimization-0: HVAC optimization strategies:\nPre-cool or pre-heat the h

## Aggregate Evaluation Report

The report calculates overall scores, strengths, weaknesses, and recommendations for improvement.

In [3]:
report = generate_evaluation_report(test_results)
print(display_evaluation_report(report))

# EcoHome Evaluation Report
Tests run: 10
Overall response score: 0.907
Overall tool completeness: 0.907

## Strengths
- The agent consistently selected the expected data tools.
- Responses were generally relevant and actionable.

## Weaknesses
- No major weaknesses observed in this test run.

## Recommendations
- Add live weather and utility-price API integrations for production use.
- Add user preference learning for comfort and EV deadline constraints.
- Add charts for hourly cost and solar-alignment recommendations.


## Example Agent Outputs

These examples show data retrieval, tool usage, citations, recommendations, and decision logs in the final answer.

In [4]:
for item in test_results[:3]:
    print('\nQUESTION:', item['question'])
    print('TOOLS:', item['tools_used'])
    print(item['final_response'][:900])


QUESTION: When should I charge my electric car tomorrow to minimize cost and maximize solar power?
TOOLS: ['search_energy_tips', 'get_weather_forecast', 'get_solar_generation', 'get_electricity_prices', 'get_energy_usage', 'calculate_savings']
EcoHome Energy Advisor recommendation
Question: When should I charge my electric car tomorrow to minimize cost and maximize solar power?

Data retrieved:
- search_energy_tips: used successfully
- get_weather_forecast: used successfully
- get_solar_generation: used successfully
- get_electricity_prices: used successfully
- get_energy_usage: used successfully
- calculate_savings: used successfully

Recommendation:
- Prioritize solar-heavy hours around 12, 11, 13:00 for flexible loads.
- Use hour 0:00 when possible at $0.13/kWh and avoid hour 16:00 at $0.39/kWh.
- Weather outlook is rainy with best solar window 11:00-13:00.
- Estimated savings for the schedule shift: $2.16, with about 4.54 kg CO2 avoided.

Relevant knowledge-base citations:
- tip_h

## Tool Coverage Table

This compact table makes it easy for reviewers to verify that each required tool is exercised.

In [5]:
coverage = []
for item in test_results:
    coverage.append({
        'id': item['id'],
        'question': item['question'],
        'tools_used': ', '.join(item['tools_used']),
        'response_score': item['response_evaluation']['overall'],
        'tool_completeness': item['tool_evaluation']['tool_completeness'],
    })
for row in coverage:
    print(row)
coverage

{'id': 'EV-001', 'question': 'When should I charge my electric car tomorrow to minimize cost and maximize solar power?', 'tools_used': 'search_energy_tips, get_weather_forecast, get_solar_generation, get_electricity_prices, get_energy_usage, calculate_savings', 'response_score': 0.969, 'tool_completeness': 1.0}
{'id': 'HVAC-002', 'question': 'What temperature should I set my thermostat on Wednesday afternoon if electricity prices spike?', 'tools_used': 'search_energy_tips, get_weather_forecast, get_solar_generation, get_electricity_prices, get_energy_usage', 'response_score': 0.856, 'tool_completeness': 1.0}
{'id': 'HISTORY-003', 'question': 'Suggest three ways I can reduce energy use based on my usage history.', 'tools_used': 'search_energy_tips, get_energy_usage', 'response_score': 0.92, 'tool_completeness': 1.0}
{'id': 'APPLIANCE-004', 'question': 'How much can I save by running my dishwasher during off-peak hours?', 'tools_used': 'search_energy_tips, get_electricity_prices, get_ene

[{'id': 'EV-001',
  'question': 'When should I charge my electric car tomorrow to minimize cost and maximize solar power?',
  'tools_used': 'search_energy_tips, get_weather_forecast, get_solar_generation, get_electricity_prices, get_energy_usage, calculate_savings',
  'response_score': 0.969,
  'tool_completeness': 1.0},
 {'id': 'HVAC-002',
  'question': 'What temperature should I set my thermostat on Wednesday afternoon if electricity prices spike?',
  'tools_used': 'search_energy_tips, get_weather_forecast, get_solar_generation, get_electricity_prices, get_energy_usage',
  'response_score': 0.856,
  'tool_completeness': 1.0},
 {'id': 'HISTORY-003',
  'question': 'Suggest three ways I can reduce energy use based on my usage history.',
  'tools_used': 'search_energy_tips, get_energy_usage',
  'response_score': 0.92,
  'tool_completeness': 1.0},
 {'id': 'APPLIANCE-004',
  'question': 'How much can I save by running my dishwasher during off-peak hours?',
  'tools_used': 'search_energy_ti